# Presentación Proyecto Minería de Datos
### Estudiantes: Lucas Purcell, Daniel Hidalgo y Sebastián Venegas

In [6]:
import pandas as pd
from pgmpy.estimators import ExhaustiveSearch
from pgmpy.estimators import BIC
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import warnings
warnings.filterwarnings("ignore")

In [7]:
df = pd.read_csv("Students Social Media Addiction.csv")


Veamos si el dataset tiene datos nulos

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 705 entries, 0 to 704
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Student_ID                    705 non-null    int64  
 1   Age                           705 non-null    int64  
 2   Gender                        705 non-null    object 
 3   Academic_Level                705 non-null    object 
 4   Country                       705 non-null    object 
 5   Avg_Daily_Usage_Hours         705 non-null    float64
 6   Most_Used_Platform            705 non-null    object 
 7   Affects_Academic_Performance  705 non-null    object 
 8   Sleep_Hours_Per_Night         705 non-null    float64
 9   Mental_Health_Score           705 non-null    int64  
 10  Relationship_Status           705 non-null    object 
 11  Conflicts_Over_Social_Media   705 non-null    int64  
 12  Addicted_Score                705 non-null    int64  
dtypes: fl

In [9]:
df_num = df.select_dtypes(include=['int64', 'float64'])
df_num = df_num.drop(columns=['Student_ID', 'Age'])
bic = BIC(df_num)
es = ExhaustiveSearch(df_num, scoring_method=bic)
best_model = es.estimate()
print(best_model.edges())

print("\nAll DAGs by score:")
for score, dag in reversed(es.all_scores()):
    print(score, dag.edges())

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Avg_Daily_Usage_Hours': 'N', 'Sleep_Hours_Per_Night': 'N', 'Mental_Health_Score': 'N', 'Conflicts_Over_Social_Media': 'N', 'Addicted_Score': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Avg_Daily_Usage_Hours': 'N', 'Sleep_Hours_Per_Night': 'N', 'Mental_Health_Score': 'N', 'Conflicts_Over_Social_Media': 'N', 'Addicted_Score': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Avg_Daily_Usage_Hours': 'N', 'Sleep_Hours_Per_Night': 'N', 'Mental_Health_Score': 'N', 'Conflicts_Over_Social_Media': 'N', 'Addicted_Score': 'N'}


[('Addicted_Score', 'Conflicts_Over_Social_Media'), ('Addicted_Score', 'Mental_Health_Score')]

All DAGs by score:
-7839.09708489312 [('Addicted_Score', 'Mental_Health_Score'), ('Conflicts_Over_Social_Media', 'Addicted_Score')]
-7839.09708489312 [('Addicted_Score', 'Conflicts_Over_Social_Media'), ('Mental_Health_Score', 'Addicted_Score')]
-7839.09708489312 [('Addicted_Score', 'Conflicts_Over_Social_Media'), ('Addicted_Score', 'Mental_Health_Score')]
-7983.742516379733 [('Mental_Health_Score', 'Addicted_Score'), ('Mental_Health_Score', 'Conflicts_Over_Social_Media')]
-7983.742516379733 [('Conflicts_Over_Social_Media', 'Mental_Health_Score'), ('Mental_Health_Score', 'Addicted_Score')]
-7983.742516379733 [('Addicted_Score', 'Mental_Health_Score'), ('Mental_Health_Score', 'Conflicts_Over_Social_Media')]
-8007.976581068707 [('Conflicts_Over_Social_Media', 'Addicted_Score'), ('Mental_Health_Score', 'Conflicts_Over_Social_Media')]
-8007.976581068707 [('Conflicts_Over_Social_Media', 'Mental_He

In [ ]:
labels = ['Low', 'Medium', 'High']

# Discretización
df['Avg_Daily_Usage_Hours_disc'] = pd.cut(df['Avg_Daily_Usage_Hours'],
                                          bins=[1.5, 3.8, 6.1, 8.5],
                                          labels=labels,
                                          include_lowest=True)

df['Sleep_Hours_Per_Night_disc'] = pd.cut(df['Sleep_Hours_Per_Night'],
                                          bins=[3.8, 5.7, 7.6, 9.6],
                                          labels=labels,
                                          include_lowest=True)

df['Mental_Health_Score_disc'] = pd.cut(df['Mental_Health_Score'],
                                        bins=[4, 6, 8, 10],
                                        labels=labels,
                                        include_lowest=True)

df['Conflicts_Over_Social_Media_disc'] = pd.cut(df['Conflicts_Over_Social_Media'],
                                                bins=[0, 2, 4, 6],
                                                labels=labels,
                                                include_lowest=True)

df['Addicted_Score_disc'] = pd.cut(df['Addicted_Score'],
                                   bins=[1, 4, 7, 10],
                                   labels=labels,
                                   include_lowest=True)

# Definir el modelo
model = DiscreteBayesianNetwork([
    ('Addicted_Score_disc', 'Conflicts_Over_Social_Media_disc'),
    ('Addicted_Score_disc', 'Mental_Health_Score_disc')
])

# Ajustar el modelo
model.fit(df, estimator=MaximumLikelihoodEstimator)

# Crear inferenciador
infer = VariableElimination(model)

# Marginalizar Mental_Health_Score_disc
marginal_mhs = infer.query(variables=['Mental_Health_Score_disc'])
print(marginal_mhs)

# Condicionar en Addicted_Score_disc = 'High'
result_high = infer.query(variables=['Mental_Health_Score_disc'], 
                          evidence={'Addicted_Score_disc': 'High'})

print(result_high)

# Condicionar en Addicted_Score_disc = 'Medium'
result_medium = infer.query(variables=['Mental_Health_Score_disc'], 
                            evidence={'Addicted_Score_disc': 'Medium'})

print(result_medium)

# Condicionar en Addicted_Score_disc = 'Low'
result_low = infer.query(variables=['Mental_Health_Score_disc'], 
                         evidence={'Addicted_Score_disc': 'Low'})

print(result_low)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Student_ID': 'N', 'Age': 'N', 'Gender': 'C', 'Academic_Level': 'C', 'Country': 'C', 'Avg_Daily_Usage_Hours': 'N', 'Most_Used_Platform': 'C', 'Affects_Academic_Performance': 'C', 'Sleep_Hours_Per_Night': 'N', 'Mental_Health_Score': 'N', 'Relationship_Status': 'C', 'Conflicts_Over_Social_Media': 'N', 'Addicted_Score': 'N', 'Avg_Daily_Usage_Hours_disc': 'O', 'Sleep_Hours_Per_Night_disc': 'O', 'Mental_Health_Score_disc': 'O', 'Conflicts_Over_Social_Media_disc': 'O', 'Addicted_Score_disc': 'O'}


+----------------------------------+---------------------------------+
| Mental_Health_Score_disc         |   phi(Mental_Health_Score_disc) |
+==================================+=================================+
| Mental_Health_Score_disc(High)   |                          0.0014 |
+----------------------------------+---------------------------------+
| Mental_Health_Score_disc(Low)    |                          0.5972 |
+----------------------------------+---------------------------------+
| Mental_Health_Score_disc(Medium) |                          0.4014 |
+----------------------------------+---------------------------------+
+----------------------------------+---------------------------------+
| Mental_Health_Score_disc         |   phi(Mental_Health_Score_disc) |
+==================================+=================================+
| Mental_Health_Score_disc(High)   |                          0.0000 |
+----------------------------------+---------------------------------+
| Ment